# Build scRNAseq reference adata object (Hurskainen et al.)

This notebook processes the scRNAseq data from Hurskainen et al. (https://www.nature.com/articles/s41467-021-21865-2), which is then used for label prediction in [01_label_prediction_scanvi.ipynb](01_label_prediction_scanvi.ipynb)

The following umi and cell metadata files were downloaded from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE151974
- GSE151974_raw_umi_matrix_postfilter.csv.gz 
- GSE151974_cell_metadata_postfilter.csv.gz 

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../conda_envs/space2_20250604.yml)

In [ ]:
import sys
import os

import pandas as pd
import anndata as ad
import scanpy as sc

## Local file info

**Make sure to set** `DATA_DIR` in `config/paths.py` to point to the location of the downloaded Xenium outputs.

The following Hurskainen et al. scRNAseq reference umi and cell metadata files **must be downloaded manually**:
- `GSE151974_raw_umi_matrix_postfilter.csv.gz`  
- `GSE151974_cell_metadata_postfilter.csv.gz`  
Available from GEO here:  
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE151974
**After downloading, set** `REF_DIR` in `config/paths.py`.

In [ ]:
from pathlib import Path
sys.path.append(str(Path.cwd().resolve().parents[0]))
from config.paths import DATA_DIR, BASE_OUTDIR, REF_DIR

# data input folder - set DATA_DIR in config/paths.py to the location of the downloaded Xenium outputs
data_input_folder = DATA_DIR 
if not os.path.exists(data_input_folder):
    os.makedirs(data_input_folder)

# reference inputs - set REF_DIR in config/paths.py to where the downloaded reference data is saved
ref_folder = REF_DIR
umi_filepath = os.path.join(ref_folder, 'GSE151974_raw_umi_matrix_postfilter.csv.gz')
cells_filepath = os.path.join(ref_folder, 'GSE151974_cell_metadata_postfilter.csv.gz')

In [ ]:
# folder to save outputs
out_dir = os.path.join(BASE_OUTDIR, "data_import/outputs_hurskainen_ref")
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

## Functions

In [ ]:
def h5_to_adata(dir, sample_ID):
    """
    Load a spatial transcriptomics dataset from an h5 and corresponding cell metadata file into an AnnData object.

    Parameters
    ----------
    dir : str
        Path to the directory containing:
        - 'cell_feature_matrix.h5': the gene expression matrix in 10x HDF5 format.
        - 'cells.csv.gz': a gzipped CSV file with cell-level metadata, including spatial coordinates.
    
    sample_ID : str
        A string identifying the sample, added to the `.obs` dataframe of the AnnData object.

    Returns
    -------
    adata : anndata.AnnData
        The annotated single-cell data object with spatial coordinates and sample ID included.
    """
    
    # file names    
    h5_file = os.path.join(dir, 'cell_feature_matrix.h5')
    cells_file = os.path.join(dir, 'cells.csv.gz')
    
    # h5 to adata
    adata = sc.read_10x_h5(
        filename=h5_file
    )
    
    # cells file
    df = pd.read_csv(
        cells_file, 
        compression = 'gzip'
    )
    
    # set indx
    df.set_index(adata.obs_names, inplace=True)
    adata.obs = df.copy()
    
    # x, y
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()

    # add sample
    adata.obs['sample'] = sample_ID
    
    return adata

## Create adata from Xenium sample
Load one of the Xenium samples to use to subset the reference adata object gene list. 

In [ ]:
xen_dir = os.path.join(data_input_folder,'output-XETG00195__0037002__TIS08781-003-003__20241016__230427')
adata_sample = h5_to_adata(xen_dir, 'TIS08781')

## Create reference adata object

In [ ]:
adata_ref = ad.read_csv(umi_filepath)
adata_ref = adata_ref.transpose()
cell_metadata_df = pd.read_csv(
    cells_filepath, 
    compression = 'gzip', index_col=0
)
assert adata_ref.obs_names.equals(cell_metadata_df.index), "ref cells and metadata are misaligned"
adata_ref.obs = cell_metadata_df

## Process reference adata

### First subset to xenium genes

In [ ]:
# subset to xenium genes
sample_genes_bool = adata_ref.var.index.isin(adata_sample.var.index)
adata_ref_xen = adata_ref[:, sample_genes_bool]

In [ ]:
print(adata_ref_xen.shape)

### Next, normalize, log, and reduce dimensions

In [ ]:
def scanpy_process(adata):
    """
    Perform standard Scanpy preprocessing on an AnnData object, assuming QC and filtering are already done.

    Parameters
    ----------
    adata : anndata.AnnData
        The AnnData object to process. Assumes that basic filtering and QC have already been performed.

    Returns
    -------
    adata : anndata.AnnData
        The same AnnData object with added layers, embeddings, and clustering results.
    """

    # norm and log
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    
    ## dimensionality reduction 
    sc.pp.pca(adata)
    print('starting neighbors...')
    sc.pp.neighbors(adata)
    print('starting umap...')
    sc.tl.umap(adata)
    # leiden
    print('starting leiden res1')
    sc.tl.leiden(adata, resolution = 1, key_added='leiden_res1', n_iterations=2)
    return adata

In [ ]:
adata_ref_xen = scanpy_process(adata_ref_xen)

In [ ]:
adata_ref_xen.obs.columns

In [ ]:
sc.pl.embedding(adata_ref_xen, basis='X_umap', color='CellType', frameon=False) 
               

## Save locally

In [ ]:
filename = os.path.join(out_dir, 'adata_ref_xen.h5ad')
adata_ref_xen.write_h5ad(filename, compression='gzip')

In [ ]:
import session_info
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show(excludes=['google3'])